# 02 · Validación manual de atributos (muestra de 200 prendas)

Paso 4 de la etapa 3. Responde a la crítica del profesor sobre datos sin verificar: una persona compara los atributos que sacamos (del nombre, de la categoría o de la foto) con la foto real del producto, y se mide la **precisión** de cada atributo con su intervalo de confianza del 95 % (Wilson).

**Muestra:** 25 prendas por slot (8 slots → 200) entre las prendas de outfits vestibles **que tienen foto** (hace falta verla para juzgar). Semilla fija, así la muestra es la misma cada vez.

**Entradas:** `polyvore_items_atributos.csv` (de `01_atributos`) y `color_imagen_polyvore.csv` (de Colab, para saber dónde está cada foto en Kaggle).
**Salidas:** `data/revision/polyvore_muestra_revision.xlsx` (planilla para revisar), `data/revision/fotos_muestra/` (200 fotos, no se suben a git) y `results/hito3/validacion_atributos.csv`.

**Cómo llenar la planilla:** en cada columna amarilla `ok_<atributo>` elegir `si` (correcto) o `no` (incorrecto); `-` = desconocido o no aplica, no se cuenta. Si es `no`, se puede escribir el valor correcto en `correccion`. Luego volver a ejecutar este notebook: las respuestas ya escritas no se borran.

**Regla escrita antes de revisar:** se juzga con la foto y el nombre. *Familia de color:* neutro = negro, blanco, crema, gris, beige, marrón, denim/navy y metales en joyería; suave = pastel o apagado claro; fuerte = el resto. *Patrón liso* es correcto si no se ve estampado. *Estilo* solo se marca `no` si claramente no corresponde.

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("No se encontró data/raw. Abre el notebook dentro del repositorio.")
sys.path.insert(0, str(ROOT / "src"))

from modelo_fly.muestra import A_REVISAR, descargar_fotos, escribir_planilla, muestra_estratificada, precision

PROC = ROOT / "data" / "processed" / "hito3"
REVISION = ROOT / "data" / "revision" / "polyvore_muestra_revision.xlsx"
FOTOS = ROOT / "data" / "revision" / "fotos_muestra"
RESULTADO = ROOT / "results" / "hito3" / "validacion_atributos.csv"
POR_SLOT, SEMILLA = 25, 2026
# Token de Kaggle: carpeta FUERA del repositorio (nunca dentro de modelo-fly).
os.environ.setdefault("KAGGLE_CONFIG_DIR", str(ROOT.parent))

items = pd.read_csv(PROC / "polyvore_items_atributos.csv", dtype=str, keep_default_na=False)
color = pd.read_csv(PROC / "color_imagen_polyvore.csv", dtype=str, keep_default_na=False, usecols=["item_key", "carpeta"])
todos = pd.read_csv(PROC / "polyvore_items.csv", dtype=str, usecols=["item_key", "tid"])
# La copia de Kaggle guarda cada producto una vez: se busca su foto por producto (tid).
fotos = (color.merge(todos, on="item_key").drop_duplicates("tid")
         .rename(columns={"item_key": "clave_foto", "carpeta": "carpeta_foto"}))
con_foto = items.merge(fotos, on="tid")
print("Prendas con foto:", len(con_foto), "de", len(items))

Prendas con foto: 60486 de 81372


## 1. Muestra y fotos

In [2]:
muestra = muestra_estratificada(con_foto, POR_SLOT, SEMILLA)
print(muestra["slot"].value_counts().to_dict())
print("Origen del tono en la muestra:", muestra["origen_tono"].where(muestra["origen_tono"] == "foto", "texto u otro").value_counts().to_dict())

fallos = descargar_fotos(muestra, FOTOS)
print("Fotos en la carpeta:", sum(1 for _ in FOTOS.glob("*.jpg")), "| fallos:", len(fallos))
for f in fallos[:5]:
    print("  ", f)

{'calzado': 25, 'bolso': 25, 'abajo': 25, 'arriba': 25, 'vestido': 25, 'joyeria': 25, 'exterior': 25, 'accesorio': 25}


Origen del tono en la muestra: {'foto': 141, 'texto u otro': 59}


Fotos en la carpeta: 200 | fallos: 0


## 2. Planilla

In [3]:
previa = pd.read_excel(REVISION, dtype=str, keep_default_na=False) if REVISION.exists() else None
planilla = escribir_planilla(muestra[["item_key", "clave_foto", "name", "categoria", *A_REVISAR, "origen_tono"]],
                             FOTOS, REVISION, previa)
marcadas = planilla[[f"ok_{a}" for a in A_REVISAR]].isin(["si", "no"]).any(axis=1).sum()
print("Planilla:", REVISION.relative_to(ROOT), "| filas", len(planilla), "| filas con alguna respuesta:", marcadas)

Planilla: data\revision\polyvore_muestra_revision.xlsx | filas 200 | filas con alguna respuesta: 0


## 3. Precisión de cada atributo

Se calcula con las respuestas `si`/`no`. Mientras la planilla no esté llena, la tabla sale vacía.

In [4]:
tabla = precision(planilla)
display(tabla.set_index("atributo"))

ok_tono = planilla["ok_tono"].str.lower()
por_origen = (planilla.assign(origen=planilla["origen_tono"].where(planilla["origen_tono"] == "foto", "texto"))
              [ok_tono.isin(["si", "no"])].groupby("origen")["ok_tono"]
              .agg(revisadas="size", precision_pct=lambda s: round(100 * s.str.lower().eq("si").mean(), 1)))
print("Tono según de dónde salió:")
display(por_origen)

RESULTADO.parent.mkdir(parents=True, exist_ok=True)
tabla.to_csv(RESULTADO, index=False, encoding="utf-8-sig", sep=";")
print("Guardado:", RESULTADO.relative_to(ROOT))

,revisadas,correctas,precision_%,ic95_bajo_%,ic95_alto_%,sin_marcar
atributo,,,,,,
slot,0,0,NaN,NaN,NaN,200
tono,0,0,NaN,NaN,NaN,200
familia_color,0,0,NaN,NaN,NaN,200
patron,0,0,NaN,NaN,NaN,199
material,0,0,NaN,NaN,NaN,69
estilo,0,0,NaN,NaN,NaN,30
calce,0,0,NaN,NaN,NaN,20
largo,0,0,NaN,NaN,NaN,23


Tono según de dónde salió:


,revisadas,precision_pct
origen,,


Guardado: results\hito3\validacion_atributos.csv
